In [1]:
using LowLevelFEM, LinearAlgebra

In [2]:
p = 3

3

In [3]:
structured_box_mesh(n=5, order=p)

#openGeometry("box2.geo")
#openPreProcessor()

mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, field=:u, reducedOrder=true);

In [4]:
k = mat.k

s = ScalarField(U, "right", (x, y, z)->y > 0.5 ? 1 : 10)

K = ∫(SymGrad(U) ⋅ [2 1 1 0 0 0; 1 2 1 0 0 0; 1 1 2 0 0 0; 0 0 0 1 0 0; 0 0 0 0 1 0; 0 0 0 0 0 1] ⋅ SymGrad(U))
f = ∫(U ⋅ [s, 0, 0], Γ="right")

bc = BoundaryCondition("left", ux=(x, y, z) -> (y^2 + z)/10, uy=0, uz=0);

In [5]:
fixed = constrainedDoFs(U, [bc])
free = freeDoFs(U, [bc]);

In [6]:
u1 = applyBoundaryConditions(U, [bc])
f_kin = K.A[:, fixed] * u1.a[fixed, 1]
u1.a[free] = (K.A[free, free]) \ (f.a[free, 1] - f_kin[free, 1]);

In [7]:
showDoFResults(u1, name="u1", visible=true);

In [8]:
T, R = reductionMatrices(U);

In [9]:
u2 = applyBoundaryConditions(U, [bc])
f_kin = K.A[:, fixed] * u2.a[fixed, 1]
Kr = T[free, :]' * K.A[free, free] * T[free, :]
fr = T[free, :]' * (f.a[free, 1] - f_kin[free, 1])

ur = Kr \ fr

u2.a[free] = (T*ur)[free];

In [10]:
u3 = solveField(K, f, support=[bc])

nodal VectorField
[0.10000000000000034; 0.0; … ; 3.4503773501925017; -0.08935431205647533;;]

In [11]:
showDoFResults(u2, name="u2", visible=true);
showDoFResults(u3, name="u3", visible=true);

In [12]:
norm(u2.a - u1.a) / norm(u1.a)

0.0037072907387662806

In [13]:
norm(u2.a - u3.a) / norm(u2.a)

0.002511679903573326

In [14]:
norm(u1.a - u3.a) / norm(u3.a)

0.005345825414739719

In [15]:
∫(U, "body", u1[1])

1.9709623591885572

In [16]:
∫(U, "body", u2[1])

1.9681699617236919

In [17]:
∫(U, "body", u3[1])

1.9652093135702569

In [19]:
T, R = reductionMatrices(U)

u3r = R * u3.a[:, 1]

norm(T * u3r - u3.a[:, 1]) / norm(u3.a[:, 1])

6.814056532895776e-16

In [20]:
u2r = R * u2.a[:, 1]

norm(T * u2r - u2.a[:, 1]) / norm(u2.a[:, 1])

0.0011607028676150028

In [18]:
openPostProcessor();

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
